# 面试问题：RAG Parent-Child Chunk 怎样兼顾小块召回与大块上下文，并避免 ACL 泄漏？

        ## 可直接复述的回答主线

        1. Parent-Child Chunk 用较小 child 做检索，用其 parent 或相邻窗口补充完整上下文。
2. 只返回 top child 容易命中标题或问题词，却丢失位于相邻句子的条件、时效和例外。
3. 底层实现应保留 parent_id、child_index、ACL、版本和字符预算，并展示 child 排名与 hydration。
4. 多个 child 命中同一 parent 时要去重，不能重复占用上下文预算。
5. ACL 必须在 child 排名之前过滤；先检索后补 parent 可能把受限父文档泄漏给 guest。
6. 生产系统还需 tokenizer 精确预算、重排、版本失效、引用 span 和权限缓存一致性。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是六篇企业知识父文档，每篇拆成标题式 child 与答案式 child，覆盖退款、续费、发票、物流、账户和受限差旅政策。五条公开查询加一条 guest 受限查询用于评估答案短语与 ACL；数据均为脱敏离线政策。

In [1]:
import re  # 对中文字符和空格业务词执行透明检索切分。
parents = [{"id": "p-refund", "title": "企业订单退款时效", "acl": "public", "version": 3, "children": ["企业订单退款时效说明", "审核通过后通常三个工作日原路到账", "高峰期以支付渠道状态为准"]}, {"id": "p-renew", "title": "会员自动续费", "acl": "public", "version": 2, "children": ["会员自动续费关闭入口", "在设置页面关闭后于下个周期生效", "已扣款订单需单独申请退款"]}, {"id": "p-invoice", "title": "电子发票抬头", "acl": "public", "version": 4, "children": ["电子发票抬头修改规则", "开票前可以修改公司名称和税号", "开票后需要先红冲再重开"]}, {"id": "p-logistics", "title": "物流异常", "acl": "public", "version": 5, "children": ["物流状态长时间未更新", "超过四十八小时可提交催件", "生鲜订单需联系专线客服"]}, {"id": "p-account", "title": "账户锁定", "acl": "public", "version": 2, "children": ["账户锁定后的恢复步骤", "完成实名验证后可以重置登录状态", "客服不会索取密码或验证码"]}, {"id": "p-travel", "title": "内部差旅额度", "acl": "finance-admin", "version": 7, "children": ["内部差旅额度政策", "总监级单次差旅额度为五千元", "超额申请需要财务负责人审批"]}]  # 定义六篇具有 ACL、版本和三段 child 的父文档。
queries = [{"id": "pc-01", "question": "企业退款多久到账", "expected_parent": "p-refund", "expected_phrase": "三个工作日", "acl": "public"}, {"id": "pc-02", "question": "关闭自动续费何时生效", "expected_parent": "p-renew", "expected_phrase": "下个周期", "acl": "public"}, {"id": "pc-03", "question": "发票公司名称什么时候能改", "expected_parent": "p-invoice", "expected_phrase": "开票前", "acl": "public"}, {"id": "pc-04", "question": "物流多久不更新可以催件", "expected_parent": "p-logistics", "expected_phrase": "四十八小时", "acl": "public"}, {"id": "pc-05", "question": "账户锁定后如何恢复", "expected_parent": "p-account", "expected_phrase": "实名验证", "acl": "public"}, {"id": "pc-06", "question": "总监差旅额度是多少", "expected_parent": None, "expected_phrase": None, "acl": "guest"}]  # 定义五条公开问题和一条无权限受限问题。
children = [{"id": f"{parent['id']}-c{index}", "parent_id": parent["id"], "index": index, "text": text, "acl": parent["acl"], "version": parent["version"]} for parent in parents for index, text in enumerate(parent["children"])]  # 展开父文档为十八个可检索 child。
parent_by_id = {parent["id"]: parent for parent in parents}  # 建立 parent_id 到完整文档的索引。
def terms(text):  # 提取连续中文、空格词和单字用于透明打分。
    chunks = re.findall(r"[A-Za-z0-9]+|[\u4e00-\u9fff]+", text.lower())  # 提取连续词段。
    characters = [character for character in text if "\u4e00" <= character <= "\u9fff"]  # 补充中文单字。
    return set(chunks + characters)  # 返回去重词元集合。
print("教学实验输入：Parent-Child 企业知识")  # 标记下方为脱敏离线政策。
for parent in parents:  # 逐父文档展示 ACL、版本和 child 数量。
    print(f"{parent['id']:<12} acl={parent['acl']:<13} v={parent['version']} children={parent['children']}")  # 输出当前父文档完整结构。
print("query count=", len(queries), "child count=", len(children))  # 展示查询和检索单元规模。

教学实验输入：Parent-Child 企业知识
p-refund     acl=public        v=3 children=['企业订单退款时效说明', '审核通过后通常三个工作日原路到账', '高峰期以支付渠道状态为准']
p-renew      acl=public        v=2 children=['会员自动续费关闭入口', '在设置页面关闭后于下个周期生效', '已扣款订单需单独申请退款']
p-invoice    acl=public        v=4 children=['电子发票抬头修改规则', '开票前可以修改公司名称和税号', '开票后需要先红冲再重开']
p-logistics  acl=public        v=5 children=['物流状态长时间未更新', '超过四十八小时可提交催件', '生鲜订单需联系专线客服']
p-account    acl=public        v=2 children=['账户锁定后的恢复步骤', '完成实名验证后可以重置登录状态', '客服不会索取密码或验证码']
p-travel     acl=finance-admin v=7 children=['内部差旅额度政策', '总监级单次差旅额度为五千元', '超额申请需要财务负责人审批']
query count= 6 child count= 18


## 2. Baseline / 基线：检索 top child 并只返回这一小段

小块能精准命中标题，但答案条件常在相邻 child。基线还先对所有 ACL 排名，使 guest 查询可能直接看到受限差旅 child。

In [2]:
def child_score(question, child):  # 计算查询与一个 child 的去重字符重叠。
    return len(terms(question) & terms(child["text"]))  # 返回透明相关性分。
baseline_rows = []  # 保存六条查询的 top child 和答案覆盖。
for query in queries:  # 对每条查询不做 ACL 过滤直接排名。
    ranking = sorted([(child_score(query["question"], child), child) for child in children], key=lambda item: (-item[0], item[1]["id"]))  # 按相关性和 child ID稳定排序。
    score, top_child = ranking[0]  # 读取最高相关小块。
    phrase_found = query["expected_phrase"] is not None and query["expected_phrase"] in top_child["text"]  # 检查单个 child 是否包含答案短语。
    leaked = query["acl"] == "guest" and top_child["acl"] != "public"  # 判断 guest 是否命中受限 child。
    baseline_rows.append({"id": query["id"], "child": top_child["id"], "parent": top_child["parent_id"], "score": score, "context": top_child["text"], "correct": phrase_found if query["expected_phrase"] is not None else not leaked, "leaked": leaked})  # 保存上下文覆盖和权限结果。
print("Baseline top child 结果")  # 标记当前输出没有 parent hydration。
print("请求      top_child          score  答案/安全正确  ACL泄漏  context")  # 输出基线结果表头。
for row in baseline_rows:  # 逐查询展示小块上下文。
    print(f"{row['id']:<9} {row['child']:<18} {row['score']:>5} {str(row['correct']):>13} {str(row['leaked']):>8}  {row['context']}")  # 输出当前查询的 top child 和失败类型。

Baseline top child 结果
请求      top_child          score  答案/安全正确  ACL泄漏  context
pc-01     p-refund-c0            4         False    False  企业订单退款时效说明
pc-02     p-renew-c0             6         False    False  会员自动续费关闭入口
pc-03     p-invoice-c1           6          True    False  开票前可以修改公司名称和税号
pc-04     p-logistics-c0         4         False    False  物流状态长时间未更新
pc-05     p-account-c0           7         False    False  账户锁定后的恢复步骤
pc-06     p-travel-c1            6         False     True  总监级单次差旅额度为五千元


## 3. 底层实现：ACL-first child 排名、parent 去重与预算 hydration

先按调用者 ACL 过滤 child，再取得 top-k。命中 child 映射回 parent，按首次得分去重并在字符预算内加入完整父文档；每个 parent 保留版本和命中 child。

In [3]:
def allowed(child, caller_acl):  # 判断调用者是否可以检索当前 child。
    return child["acl"] == "public" or child["acl"] == caller_acl  # public 对所有人可见，受限文档要求 ACL 精确匹配。
def parent_child_retrieve(query, top_k=3, character_budget=180):  # 执行小块检索和父文档 hydration。
    candidates = [child for child in children if allowed(child, query["acl"])]  # 在打分前过滤不可见 child。
    ranking = sorted([{"score": child_score(query["question"], child), "child": child} for child in candidates], key=lambda item: (-item["score"], item["child"]["id"]))  # 对可见 child 排名。
    selected_children = [row for row in ranking[:top_k] if row["score"] > 0]  # 只保留有词元重叠的前三个 child。
    hydrated = []  # 保存去重后的父上下文。
    seen_parents = set()  # 防止同一 parent 多个 child 重复占预算。
    used_characters = 0  # 记录已使用字符预算。
    for row in selected_children:  # 按 child 得分顺序尝试 hydration。
        parent_id = row["child"]["parent_id"]  # 读取当前 child 所属父文档。
        if parent_id in seen_parents:  # 同一 parent 只返回一次。
            continue  # 跳过重复父文档。
        parent = parent_by_id[parent_id]  # 读取完整父文档和版本。
        parent_text = "；".join(parent["children"])  # 拼接父文档全部 child 形成完整上下文。
        if used_characters + len(parent_text) > character_budget:  # 检查加入父文档后是否超过字符预算。
            continue  # 超预算时跳过当前 parent。
        hydrated.append({"parent": parent, "text": parent_text, "matched_child": row["child"]["id"], "score": row["score"]})  # 保存父上下文与检索证据。
        seen_parents.add(parent_id)  # 标记当前 parent 已加入。
        used_characters += len(parent_text)  # 累加字符预算。
    return ranking, hydrated, used_characters  # 返回 child 排名、父上下文和预算用量。
first_ranking, first_hydrated, first_budget = parent_child_retrieve(queries[0])  # 对退款问题展示完整检索过程。
print("pc-01 child 排名前五")  # 标记下表展示小块召回分项。
print("child              parent       score  text")  # 输出 child 排名表头。
for row in first_ranking[:5]:  # 展示前五个可见 child。
    print(f"{row['child']['id']:<18} {row['child']['parent_id']:<12} {row['score']:>5}  {row['child']['text']}")  # 输出当前 child 得分和文本。
print("hydrated=", [(row["parent"]["id"], row["matched_child"], row["text"]) for row in first_hydrated], "budget=", first_budget)  # 展示 parent hydration 与实际预算。

pc-01 child 排名前五
child              parent       score  text
p-refund-c0        p-refund         4  企业订单退款时效说明
p-refund-c1        p-refund         2  审核通过后通常三个工作日原路到账
p-renew-c2         p-renew          2  已扣款订单需单独申请退款
p-account-c0       p-account        1  账户锁定后的恢复步骤
p-account-c1       p-account        0  完成实名验证后可以重置登录状态
hydrated= [('p-refund', 'p-refund-c0', '企业订单退款时效说明；审核通过后通常三个工作日原路到账；高峰期以支付渠道状态为准'), ('p-renew', 'p-renew-c2', '会员自动续费关闭入口；在设置页面关闭后于下个周期生效；已扣款订单需单独申请退款')] budget= 79


## 4. 结果表与结果解读

对公开问题检查 hydrated context 是否包含期望答案；对 guest 受限问题检查是否返回任何差旅 parent。小块负责定位，父块负责补足条件。

In [4]:
corrected_rows = []  # 保存六条查询的 parent-child 结果。
for query in queries:  # 逐查询执行 ACL-first hydration。
    ranking, hydrated, used_characters = parent_child_retrieve(query)  # 获取 child 排名和去重父上下文。
    context = "\n".join(row["text"] for row in hydrated)  # 合并预算内父上下文。
    selected_parents = [row["parent"]["id"] for row in hydrated]  # 记录最终返回的 parent ID。
    if query["expected_phrase"] is None:  # 受限 guest 查询的正确行为是不返回目标受限 parent。
        correct = "p-travel" not in selected_parents  # 检查受限 parent 没有泄漏。
    else:  # 公开知识问题需要答案短语和正确 parent。
        correct = query["expected_phrase"] in context and query["expected_parent"] in selected_parents  # 检查父上下文覆盖完整答案。
    corrected_rows.append({"id": query["id"], "parents": selected_parents, "context": context, "budget": used_characters, "correct": correct})  # 保存逐查询结果。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(queries)  # 计算 child-only 的答案与安全正确率。
corrected_accuracy = sum(row["correct"] for row in corrected_rows) / len(queries)  # 计算 parent-child 的答案与安全正确率。
print("请求      Baseline正确  Parent-Child正确  hydrated parents          budget")  # 输出同查询对照表头。
for baseline, corrected in zip(baseline_rows, corrected_rows):  # 逐查询比较小块和父块。
    print(f"{corrected['id']:<9} {str(baseline['correct']):>12} {str(corrected['correct']):>18} {str(corrected['parents']):<27} {corrected['budget']:>6}")  # 输出当前查询的结果和预算。
print(f"结果解读：child-only正确率={baseline_accuracy:.1%}，Parent-Child={corrected_accuracy:.1%}；提升来自 sibling 条件补全和 ACL-first。")  # 解释同一批查询的差异。

请求      Baseline正确  Parent-Child正确  hydrated parents          budget
pc-01            False               True ['p-refund', 'p-renew']         79
pc-02            False               True ['p-renew', 'p-refund']         79
pc-03             True               True ['p-invoice', 'p-account']      76
pc-04            False               True ['p-logistics', 'p-account']     74
pc-05            False               True ['p-account', 'p-refund']       79
pc-06            False               True []                               0
结果解读：child-only正确率=16.7%，Parent-Child=100.0%；提升来自 sibling 条件补全和 ACL-first。


## 5. 失败案例与修正

guest 查询“总监差旅额度”时，未过滤排名会命中 finance-admin child。修正必须在 child 打分前应用 ACL，而不是 hydrate 后再删文本。

In [5]:
guest_baseline = next(row for row in baseline_rows if row["id"] == "pc-06")  # 读取 guest 的不安全 child-only 结果。
guest_corrected = next(row for row in corrected_rows if row["id"] == "pc-06")  # 读取 ACL-first parent-child 结果。
print(f"错误行为：top_child={guest_baseline['child']} parent={guest_baseline['parent']} leaked={guest_baseline['leaked']} context={guest_baseline['context']}")  # 展示受限 child 在检索阶段泄漏。
print(f"修正行为：hydrated_parents={guest_corrected['parents']}，包含p-travel={'p-travel' in guest_corrected['parents']}，context={guest_corrected['context']}")  # 展示 ACL 过滤后的安全结果。

错误行为：top_child=p-travel-c1 parent=p-travel leaked=True context=总监级单次差旅额度为五千元
修正行为：hydrated_parents=[]，包含p-travel=False，context=


## 6. 生产边界

字符预算应替换为真实 tokenizer Token 预算，并加入 cross-encoder 重排、相邻窗口策略、版本失效、parent 摘要、引用 span、租户 ACL 缓存和删除传播。大 parent 也可能引入噪声，需要离线评测。

In [6]:
diagnostics = {"queries": len(queries), "children": len(children), "parents": len(parents), "baseline_accuracy": baseline_accuracy, "parent_child_accuracy": corrected_accuracy, "acl_leaks_after_fix": sum("p-travel" in row["parents"] for row in corrected_rows if row["id"] == "pc-06")}  # 汇总检索质量、规模和安全指标。
print("生产监控快照：", diagnostics)  # 输出 Parent-Child 服务应持续跟踪的指标。

生产监控快照： {'queries': 6, 'children': 18, 'parents': 6, 'baseline_accuracy': 0.16666666666666666, 'parent_child_accuracy': 1.0, 'acl_leaks_after_fix': 0}


## 7. 最小回归测试

只验证样本规模、parent 去重、答案补全、ACL 阻断和总体结果。

In [7]:
assert len(parents) >= 5 and len(queries) >= 5  # 保证案例有足够父文档和真实语义查询。
assert len({row["parent"]["id"] for row in first_hydrated}) == len(first_hydrated)  # 保证同一 parent 不会重复占上下文预算。
assert next(row for row in corrected_rows if row["id"] == "pc-01")["correct"]  # 保证退款 sibling 中的答案被 parent hydration 补全。
assert guest_baseline["leaked"] and "p-travel" not in guest_corrected["parents"]  # 保证失败案例复现且 ACL 修正有效。
assert corrected_accuracy > baseline_accuracy  # 保证同一批查询上的答案与安全结果优于 child-only。